Język: [English](lab.en.ipynb) | **Polski**

# Laboratorium 11. Pipeline inferencji niewielkiego modelu językowego

Materiał towarzyszący [`README.pl.md`](README.pl.md) prowadzi przez cały pipeline:
prompt, tokenizację, model, generowanie, dekodowanie, pomiar i ocenę.

**Sekcje 0-11 tworzą obowiązkowy rdzeń na około 105 minut. Pozostałe 15 minut
stanowi bufor.** Sekcje R1-R2 na końcu notebooka są opcjonalne.

In [ ]:
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(0)
torch.set_num_threads(4)
device = torch.device("cpu")
print(f"Uzywane urzadzenie: {device}")

MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"
REVISION = "12fd25f77366fa6b3b4b768ec3050bf629380bac"

## 1. Model i tokenizer

In [ ]:
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REVISION, dtype=torch.float32)
model.eval()
print(f"load time: {time.time() - t0:.2f}s")
print(f"liczba parametrow: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# TODO: wypisz model.config.num_attention_heads i model.config.num_key_value_heads
# Czy sa rowne? Co to oznacza dla kosztu pamieci KV cache? (zobacz wyklad)
raise NotImplementedError("TODO: sprawdzenie glow Q vs KV")

## 2. Przygotowanie promptów

Należy przygotować **dokładnie 3 obowiązkowe** prompty:
1. **jeden** prompt jednoznaczny typu "ekstrakcja informacji" (ma precyzyjną
   oczekiwaną odpowiedź),
2. **jeden drugi** prompt jednoznaczny, innego typu (np. "przekształcenie
   formatu"),
3. **jeden** prompt **jakościowy** (parafraza albo kontynuacja, `expected=None`).

Dla **każdego** promptu z jednoznaczną odpowiedzią trzeba zapisać tę odpowiedź
**przed uruchomieniem modelu**. Wymagana struktura to lista słowników
`{"id", "typ", "prompt", "expected"}`.

**Rozszerzenie R1 (opcjonalne):** można dodać 2 kolejne prompty, na przykład
pytanie z kontekstem i kontynuację, a następnie wykonać pełny zakres z rdzenia.

In [ ]:
# TODO: zbuduj liste DOKLADNIE 3 obowiazkowych slownikow {"id","typ","prompt","expected"}:
# 1. jeden prompt jednoznaczny (np. ekstrakcja informacji), expected != None
# 2. jeden DRUGI prompt jednoznaczny, innego typu (np. przeksztalcenie formatu)
# 3. jeden prompt JAKOSCIOWY (parafraza albo kontynuacja), expected=None
prompts = []
raise NotImplementedError("TODO: przygotowanie promptow")

## 3. Szablon rozmowy

Funkcja `format_prompt(prompt_text)` powinna zwracać `inputs` gotowe do
`model.generate(**inputs, ...)`, używając `tokenizer.apply_chat_template()`
z `add_generation_prompt=True`. **Nie** dodawaj ręcznie żadnych tokenów
specjalnych. `apply_chat_template` już je wstawia (zob. wykład, sekcja
"Chat templates").

**Uwaga dotycząca API.** `apply_chat_template(...,
tokenize=True, return_tensors="pt")` **bez** `return_dict=True` może zwrócić
inny typ zależnie od wersji `transformers`. Dlatego należy jawnie ustawić
`return_dict=True` i korzystać z `inputs["input_ids"]`, zamiast zakładać,
że wynikiem będzie tensor.

Na końcu trzeba sprawdzić, czy funkcja rzeczywiście dodaje tokeny specjalne.
Zdekodowany prompt ma zaczynać się od tokenu BOS/
`<|im_start|>`, nie od treści promptu).

In [ ]:
def format_prompt(prompt_text):
    # TODO:
    # 1. zbuduj messages = [{"role": "user", "content": prompt_text}]
    # 2. zwroc tokenizer.apply_chat_template(messages, tokenize=True,
    #    add_generation_prompt=True, return_tensors="pt", return_dict=True)
    raise NotImplementedError("TODO: format_prompt")

**Weryfikacja (gotowa asercja): zdekodowany początek sformatowanego promptu
musi zaczynać się od tokenu specjalnego (BOS/`<|im_start|>`), nie od treści.**

In [ ]:
check_inputs = format_prompt("test")  # noqa: F821 - zdefiniowane w sekcji 3
start_decoded = tokenizer.decode(check_inputs["input_ids"][0][:1])
print(f"pierwszy token sformatowanego promptu: {start_decoded!r}")
assert start_decoded == tokenizer.bos_token, (
    f"BLAD: oczekiwano {tokenizer.bos_token!r} jako pierwszy token, dostano {start_decoded!r}"
)
print("OK: format_prompt dodaje token specjalny na starcie.")

## 4. Generowanie i poprawne dekodowanie

Funkcja `generate_response(prompt_text, **gen_kwargs)` powinna:
1. sformatować prompt (sekcja 3),
2. wywołać `model.generate(**inputs, **gen_kwargs, pad_token_id=...)`,
3. **wyciągnąć TYLKO nowo wygenerowane tokeny** (nie cały `output`, który
   zawiera też prompt; zob. wykład, eksperyment B/E),
4. zdekodować wynik (`skip_special_tokens=True`),
5. zwrócić tekst **oraz** liczbę nowo wygenerowanych tokenów (potrzebne do
   pomiaru przepustowości w sekcji 6).

In [ ]:
def generate_response(prompt_text, **gen_kwargs):
    # TODO:
    # 1. inputs = format_prompt(prompt_text)
    # 2. prompt_len = inputs["input_ids"].shape[1]
    # 3. out = model.generate(**inputs, pad_token_id=tokenizer.pad_token_id, **gen_kwargs)
    # 4. WYCIAGNIJ TYLKO nowo wygenerowane tokeny: out[0, prompt_len:] (NIE caly out!)
    # 5. zdekoduj (skip_special_tokens=True)
    # 6. zwroc (text, liczba_nowych_tokenow)
    raise NotImplementedError("TODO: generate_response")

**Weryfikacja (gotowa asercja): dla `do_sample=False`, dwa wywołania z tym
samym promptem muszą dać identyczny wynik.**

In [ ]:
r1, _ = generate_response("What is the capital of France?", max_new_tokens=10, do_sample=False)  # noqa: F821
r2, _ = generate_response("What is the capital of France?", max_new_tokens=10, do_sample=False)  # noqa: F821
print(f"przebieg 1: {r1!r}")
print(f"przebieg 2: {r2!r}")
assert r1 == r2, "BLAD: greedy decoding powinien byc deterministyczny"
print("OK: deterministyczne powtorzenia sa identyczne.")

## 5. Deterministyczne generowanie i sampling

Dla **wszystkich 3 obowiązkowych** promptów z sekcji 2 należy wygenerować odpowiedź
**greedy** (`do_sample=False`). Dodatkowo, dla **podzbioru** (prompt
jednoznaczny `extract` **i** prompt jakościowy `paraphrase`, ale **nie**
`format`, aby nie wykonywać pełnego iloczynu 3 prompty × 3 strategie), trzeba
też uruchomić **sampling** z dwiema temperaturami (0,7 i 1,3). Przed każdym
wywołaniem samplingu należy ustawić to samo ziarno (`torch.manual_seed(...)`).

**Rozszerzenie R2 (opcjonalne):** sampling można uruchomić także dla `format`
oraz promptów z R1. Pozwoli to sprawdzić jego wpływ na zadania jednoznaczne
wyłączone z porównania w rdzeniu.

In [ ]:
configs_greedy = [("greedy", dict(do_sample=False))]
configs_sampling = [
    ("sampling tau=0.7", dict(do_sample=True, temperature=0.7)),
    ("sampling tau=1.3", dict(do_sample=True, temperature=1.3)),
]
SAMPLING_SUBSET = {"extract", "paraphrase"}  # id-i promptow, ktore dodatkowo dostaja sampling

# TODO:
# dla kazdego promptu (sekcja 2): zawsze greedy; DODATKOWO sampling (oba warianty
# z configs_sampling) TYLKO jesli p["id"] in SAMPLING_SUBSET
# 1. ustaw torch.manual_seed(0) PRZED kazdym wywolaniem (powtarzalnosc)
# 2. zmierz czas (time.time() przed/po) wywolania
#    generate_response(prompt, max_new_tokens=40, **kwargs)
# 3. zbierz wyniki (lista slownikow: id, strategy, text, n_new, time, tok_s)
results = []
raise NotImplementedError("TODO: generowanie - greedy dla wszystkich, sampling dla podzbioru")

## 6. Pomiar czasu i przepustowości

Czasy zostały już zebrane w sekcji 5. Na ich podstawie należy wyświetlić tabelę
liczby tokenów na sekundę dla każdej kombinacji promptu i strategii z `results`.

In [ ]:
# TODO: wypisz id, strategie i tok/s dla kazdego wpisu w `results`
raise NotImplementedError("TODO: tabela przepustowosci")

## 7. Wpływ długości promptu

Należy wybrać **jeden** prompt i przygotować jego dłuższą wersję, na przykład
przez dodanie kontekstu lub historii rozmowy. Następnie trzeba zmierzyć czas
generowania **tej samej liczby** nowych
tokenów dla krótkiej i długiej wersji promptu.

Wartości `min_new_tokens` i `max_new_tokens` muszą być równe. Wymusza to
**tę samą** liczbę nowych tokenów dla obu długości promptu. W przeciwnym razie
generowania (np. wcześniejsze zatrzymanie przez EOS) zanieczyszczą pomiar
samego wpływu długości promptu.

In [ ]:
short_prompt = "What is 2+2?"
long_prompt = ("Here is a lot of context to pad out the prompt. " * 30) + " What is 2+2?"

# TODO: dla obu promptow, zmierz czas generate_response z max_new_tokens=20 I
# min_new_tokens=20 (ta sama liczba nowych tokenow dla obu!), wypisz prompt_len i czas
raise NotImplementedError("TODO: wplyw dlugosci promptu")

## 8. Pamięć podręczna KV

In [ ]:
# TODO: dla use_cache=True i use_cache=False, zmierz czas generate_response
# ("Tell me about the solar system.", max_new_tokens=40, do_sample=False, use_cache=...)
raise NotImplementedError("TODO: KV cache")

## 9. Ocena jakości

1. **Automatycznie**, dla **obu** promptów z jednoznaczną odpowiedzią
   (sekcja 2), dla odpowiedzi **greedy**, należy sprawdzić
   `p["expected"].lower() in response.lower()`.
2. **Jakościowo**, dla promptu **jakościowego** z sekcji 2 trzeba ocenić każdą
   uruchomioną strategię (greedy + 2 warianty samplingu z sekcji 5) według
   pięciu kryteriów z README (poprawność wykonania, kompletność, zgodność z
   formatem, naturalność, powtarzalność). Ocena wymaga **konkretnego uzasadnienia
   opartego na fragmencie odpowiedzi**, a nie stwierdzenia "wygląda dobrze".

❌ "Ta odpowiedź wygląda lepiej, więc jest lepsza."

✅ Osobne kryteria pozwalają uniknąć oceny opartej na jednym, subiektywnym
wrażeniu.

**Rozszerzenie R1/R2 (opcjonalne):** jeśli wykonano R1 (dodatkowe prompty)
lub R2 (dodatkowy sampling), obowiązuje ten sam schemat oceny.

In [ ]:
# TODO: dla obu promptow z expected != None, sprawdz czy
# p["expected"].lower() in <odpowiedz greedy tego promptu>.lower()
# wypisz id, expected, wynik sprawdzenia, i sama odpowiedz
raise NotImplementedError("TODO: automatyczna ocena")

W komórce tekstowej należy ocenić każdą strategię uruchomioną dla promptu
`paraphrase` (greedy, tau=0,7 i tau=1,3). Trzeba sprawdzić zachowanie znaczenia
oryginału, kompletność oraz naturalność. Każdy wynik wymaga konkretnego
uzasadnienia, a nie ogólnego stwierdzenia "wygląda dobrze".

_(Miejsce na odpowiedź.)_

## 10. Zadanie diagnostyczne

Poniższy kod **wygląda** poprawnie. Nie zgłasza wyjątku i generuje tekst, ale
ma błąd: pomylenie `max_length` z `max_new_tokens`.

In [ ]:
DESIRED_NEW_TOKENS = 40
diag_prompt = "Write a short poem about the ocean."
diag_inputs = format_prompt(diag_prompt)  # noqa: F821
diag_prompt_len = diag_inputs["input_ids"].shape[1]
print(f"prompt_len={diag_prompt_len}, chcemy {DESIRED_NEW_TOKENS} NOWYCH tokenow")

with torch.no_grad():
    out_buggy = model.generate(
        **diag_inputs,
        max_length=DESIRED_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
actual_new_buggy = out_buggy.shape[1] - diag_prompt_len
print(
    f"BuggyGeneration: faktycznie nowych tokenow = {actual_new_buggy} "
    f"(chcielismy {DESIRED_NEW_TOKENS}!)"
)
text_buggy = tokenizer.decode(out_buggy[0, diag_prompt_len:], skip_special_tokens=True)
print(f"tekst (uciety): {text_buggy!r}")

Należy wykonać pięć kroków:
1. Uruchomić kod z krótkim promptem i sprawdzić, ile **nowych** tokenów
   faktycznie powstało, względem tego, ile **chciał** autor kodu (zmienna
   `DESIRED_NEW_TOKENS`)?
2. Wyjaśnić, co dokładnie liczy `max_length`, a co
   `max_new_tokens` (zob. wykład, sekcja "Okno kontekstowe i
   obcinanie".)
3. Uruchomić **ten sam** kod z **dłuższym** promptem, dla którego
   `prompt_len >= max_length`, i opisać wynik. Powinno wystąpić inne,
   wyraźniejsze zachowanie niż przy krótkim prompcie.
4. Poprawić kod, zastępując `max_length=` przez `max_new_tokens=`.
5. Zweryfikować poprawkę. Dla **obu** długości promptu liczba nowo
   wygenerowanych tokenów musi być zgodna z zamierzoną.

In [ ]:
# TODO:
# 1. jako komentarz: zdiagnozuj dokladnie, co max_length liczy tutaj, a co chcielismy
# 2. sprobuj TEJ SAMEJ buggy konfiguracji z DLUZSZYM promptem (prompt_len >= max_length)
#    - co sie teraz dzieje? (podpowiedz: inny, glosniejszy blad)
# 3. napraw: uzyj max_new_tokens= zamiast max_length=
# 4. zweryfikuj POPRAWKE dla OBU dlugosci promptu (assert na liczbie nowych tokenow)
raise NotImplementedError("TODO: diagnoza i poprawka max_length/max_new_tokens")

## 11. Wnioski

**1. Która strategia dekodowania dała najbardziej użyteczne wyniki w zadaniach z jednoznaczną odpowiedzią (ekstrakcja, format)? Czy sampling, tam gdzie został przetestowany, pomógł, zaszkodził czy nie miał znaczenia?**

_(Miejsce na odpowiedź.)_

**2. Czy wyższa temperatura dała subiektywnie lepsze wyniki w zadaniu jakościowym? Czy różnorodność miała znaczenie w tym przypadku?**

_(Miejsce na odpowiedź.)_

**3. Jaki jest zmierzony koszt pipeline'u? Jak zmienia się z długością promptu i z `use_cache`?**

_(Miejsce na odpowiedź.)_

**4. Jakie konkretne ograniczenia modelu lub pipeline'u ujawniły się podczas eksperymentu?**

_(Miejsce na odpowiedź.)_

**5. Jaki wniosek wynika z kompromisu między jakością, kosztem i konfiguracją? Czy "lepsza" konfiguracja zawsze oznacza lepszy wynik dla każdego typu zadania?**

_(Miejsce na odpowiedź.)_

---

## Rozszerzenia (opcjonalne, jeśli zostanie czas po sekcjach 0-11)

- **R1.** Można dodać 2 kolejne prompty, na przykład pytanie z kontekstem
  i kontynuację tekstu, a następnie wykonać dla nich te same kroki
  (sekcje 5, 6, 9) co dla promptów obowiązkowych.
- **R2.** Można uruchomić sampling z obiema temperaturami także dla `format`
  oraz promptów z R1. Wyniki pokażą, czy sampling pomaga, szkodzi czy
  nie ma znaczenia dla zadań jednoznacznych wyłączonych z porównania w rdzeniu.
- Można dodać **szósty** prompt własnego projektu, w innym stylu niż propozycje
  wyżej.
- Można zmierzyć wpływ `repetition_penalty` na kontynuację tekstu i sprawdzić,
  powtarzalność fraz przy dłuższym generowaniu?).

_(Rozszerzenie opcjonalne. Kod i wyniki nie są wymagane w rdzeniu.)_